### Este notebook irá montar um tabela com dados dos Municipios Brasileiros
### A fonte principal serão os dados do IBGE (https://geoftp.ibge.gov.br)
### 1. Código, Nome, UF e Biomada do município
### 2. Grandes regiões (Centro Oeste, Nordeste, Norte, Sudeste e Sul)

### Estes dados serão unidos e estarão disponíveis em uma tabela com a seguinte estrutura:

- Nome da tabela: A definir ->  camada.schema.nome_da_tabela
- codigo_municipio: string (nullable = true)
- nome_municipio: string (nullable = true)
- UF_municipio: string (nullable = true)
- nome_bioma: string (nullable = true)
- grande_regiao: string (nullable = true)

In [1]:
import os, sys, requests
from pathlib import Path
from urllib.parse import urlparse

# from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import sys, os


In [2]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


c:\Marco Conti\Projetos\mais_einstein\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


### Download de um arquivo CSV do IBGE contendo informações sobre o bioma predominante por município.

In [3]:


url_source_bioma = "https://geoftp.ibge.gov.br/informacoes_ambientais/estudos_ambientais/biomas/documentos/Bioma_Predominante_por_Municipio_2024.csv"

filename = Path(urlparse(url_source_bioma).path).name or "bioma.csv"
out_path = Path.cwd() / filename

resp = requests.get(url_source_bioma, stream=True)
resp.raise_for_status()
with open(out_path, "wb") as f:
    for chunk in resp.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)

print(f"Saved file to: {out_path}")

Saved file to: c:\Marco Conti\Projetos\mais_einstein\Municipio\Bioma_Predominante_por_Municipio_2024.csv


### Lê o arquivo CSV baixado em um DataFrame do Spark, inferindo o esquema e usando o ponto e vírgula como separador.
### Rename de colunas para padrão Eintein

In [4]:

df_bioma = \
    spark.read.csv(str(out_path)
                  ,header=True
                  ,sep=";"
                  ,inferSchema=True)

df_bioma = \
    df_bioma.withColumnRenamed("Geocódigo", "codigo_municipio") \
            .withColumnRenamed("Nome do município", "nome_municipio") \
            .withColumnRenamed("Sigla da UF", "UF_municipio") \
            .withColumnRenamed("Bioma Predominante", "nome_bioma")



# Teste Remover ao concluir o pipeline
# Filtrar para manter APENAS as linhas onde o código do município começa com números (Dígitos)
df_bioma = \
    df_bioma.filter("nome_bioma is not null and codigo_municipio rlike '^[0-9]+$'")

df_bioma.limit(10).show(truncate=False)


+----------------+---------------------+------------+----------+
|codigo_municipio|nome_municipio       |UF_municipio|nome_bioma|
+----------------+---------------------+------------+----------+
|1100015         |Alta Floresta D'Oeste|RO          |Amazônia  |
|1100023         |Ariquemes            |RO          |Amazônia  |
|1100031         |Cabixi               |RO          |Amazônia  |
|1100049         |Cacoal               |RO          |Amazônia  |
|1100056         |Cerejeiras           |RO          |Amazônia  |
|1100064         |Colorado do Oeste    |RO          |Amazônia  |
|1100072         |Corumbiara           |RO          |Amazônia  |
|1100080         |Costa Marques        |RO          |Amazônia  |
|1100098         |Espigão D'Oeste      |RO          |Amazônia  |
|1100106         |Guajará-Mirim        |RO          |Amazônia  |
+----------------+---------------------+------------+----------+



### Completa a lista com as Grandes Regiões do Brasil, associando cada estado à sua respectiva região.

In [5]:

grande_regiao = [
    {"UF": "AC", "grande_regiao": "Norte"},
    {"UF": "AL", "grande_regiao": "Nordeste"},
    {"UF": "AM", "grande_regiao": "Norte"},
    {"UF": "AP", "grande_regiao": "Norte"},
    {"UF": "BA", "grande_regiao": "Nordeste"},
    {"UF": "CE", "grande_regiao": "Nordeste"},
    {"UF": "DF", "grande_regiao": "Centro-Oeste"},
    {"UF": "ES", "grande_regiao": "Sudeste"},
    {"UF": "GO", "grande_regiao": "Centro-Oeste"},
    {"UF": "MA", "grande_regiao": "Nordeste"},
    {"UF": "MG", "grande_regiao": "Sudeste"},
    {"UF": "MS", "grande_regiao": "Centro-Oeste"},
    {"UF": "MT", "grande_regiao": "Centro-Oeste"},
    {"UF": "PA", "grande_regiao": "Norte"},
    {"UF": "PB", "grande_regiao": "Nordeste"},
    {"UF": "PE", "grande_regiao": "Nordeste"},
    {"UF": "PI", "grande_regiao": "Nordeste"},
    {"UF": "PR", "grande_regiao": "Sul"},
    {"UF": "RJ", "grande_regiao": "Sudeste"},
    {"UF": "RN", "grande_regiao": "Nordeste"},
    {"UF": "RO", "grande_regiao": "Norte"},
    {"UF": "RR", "grande_regiao": "Norte"},
    {"UF": "RS", "grande_regiao": "Sul"},
    {"UF": "SC", "grande_regiao": "Sul"},
    {"UF": "SE", "grande_regiao": "Nordeste"},
    {"UF": "SP", "grande_regiao": "Sudeste"},
    {"UF": "TO", "grande_regiao": "Norte"}
    ]


### Faz a junção entre Município, Bioma e Grande Região

In [6]:
df_grande_regiao = spark.createDataFrame(grande_regiao)

df_bioma_grande_regiao = \
    df_bioma.join(df_grande_regiao, df_bioma.UF_municipio == df_grande_regiao.UF, "left") \
            .drop(df_grande_regiao.UF)

df_bioma_grande_regiao.limit(100).show(truncate=False)

+----------------+------------------------+------------+----------+-------------+
|codigo_municipio|nome_municipio          |UF_municipio|nome_bioma|grande_regiao|
+----------------+------------------------+------------+----------+-------------+
|1100015         |Alta Floresta D'Oeste   |RO          |Amazônia  |Norte        |
|1100023         |Ariquemes               |RO          |Amazônia  |Norte        |
|1100031         |Cabixi                  |RO          |Amazônia  |Norte        |
|1100049         |Cacoal                  |RO          |Amazônia  |Norte        |
|1100056         |Cerejeiras              |RO          |Amazônia  |Norte        |
|1100064         |Colorado do Oeste       |RO          |Amazônia  |Norte        |
|1100072         |Corumbiara              |RO          |Amazônia  |Norte        |
|1100080         |Costa Marques           |RO          |Amazônia  |Norte        |
|1100098         |Espigão D'Oeste         |RO          |Amazônia  |Norte        |
|1100106        

### Salva o Dataframe como csv e parquet

In [ ]:
(df_bioma_grande_regiao
    .toPandas()
    .to_csv(r"C:\Marco Conti\Projetos\MAIS-v2\dados\municipios\tb_munic_sit_bioma.csv"
           ,index=False
           ,sep=";"))



(df_bioma_grande_regiao
    .toPandas()
    .to_parquet(r"C:\Marco Conti\Projetos\MAIS-v2\dados\municipios\tb_munic_sit_bioma.parquet"))



In [ ]:
df_bioma_grande_regiao.printSchema()

In [ ]:
os.remove(out_path)

In [ ]:
# df_x1 = spark.read.csv(r"C:\Marco Conti\Projetos\MAIS-v2\dados\municipios\tb_munic_sit_bioma.csv", header=True, sep=";", inferSchema=True)
# df_x1.show(10,False)

df_x2 = spark.read.parquet(r"C:\Marco Conti\Projetos\MAIS-v2\dados\municipios\tb_munic_sit_bioma.parquet")
# df_x2.printSchema()
df_x2.show(10,False)